# Colab Regression Worker

Run this notebook in a Colab GPU runtime after adding `GITHUB_TOKEN` to Colab Secrets. The first bring-up loop runs a safe backend check for 5 cycles, pulls latest GitHub code each cycle, and pushes worker manifests/logs back to GitHub.

## 1. Clone or Update Repository

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
REPO_DIR = Path("/content/Aberration-Simulation")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = os.environ.get("GITHUB_TOKEN")

if token:
    os.environ["GITHUB_TOKEN"] = token
    print("GITHUB_TOKEN loaded from Colab Secrets.")
else:
    print("WARNING: GITHUB_TOKEN is not loaded. Pull may work, but push will fail.")

if REPO_DIR.exists():
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Check GPU

In [ ]:
!nvidia-smi
!python scripts/check_backend.py

## 3. Start 5-Cycle Worker Loop

In [ ]:
!python scripts/colab_regression_worker.py --config experiments/colab_worker_5cycles.json --cycles 5